# Mini-Project 5: Taxi Route Revenue, Pricing & Fleet Planner

**Context:** A Kampala matatu association operates 14-seater vehicles on three routes. This notebook analyses the supplied ten days of passenger counts, compares transparent forecasting methods, estimates day-11 gross revenue, plans fleet size, and tests a weekly seasonal pattern using synthetic data.

**Reproducibility:** the extension uses `np.random.default_rng(2026)`. Run the notebook from top to bottom. Reusable logic lives in the `src/` package; this notebook focuses on analysis and interpretation.

**Assumptions:** passenger counts are total passenger boardings per route per day; revenue is gross fare revenue before costs; population variance/standard deviation describe the supplied ten-day period; day-11 forecast model is selected by lowest route-specific MAE over days 4–10.

In [ ]:
from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure imports work whether Jupyter is launched at the repository root or in notebooks/.
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.routes import Route
from src.equilibrium import solve_ntinda_equilibrium
from src.forecasting import (
    MovingAverageForecaster,
    ExponentialSmoothingForecaster,
    LinearTrendForecaster,
    SeasonalNaiveForecaster,
    rolling_origin_predictions,
    mean_absolute_error,
    tune_alpha,
)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
print(f"Project root: {ROOT}")

## 1. Route data, revenue and descriptive statistics

In [ ]:
passenger_data = {
    "Kampala-Ntinda": [35, 40, 42, 50, 55, 60, 48, 52, 47, 45],
    "Kampala-Entebbe": [60, 58, 65, 70, 72, 80, 75, 68, 66, 64],
    "Kampala-Mukono": [45, 47, 50, 49, 55, 62, 58, 53, 51, 50],
}
fares = {
    "Kampala-Ntinda": 2000,
    "Kampala-Entebbe": 5000,
    "Kampala-Mukono": 3000,
}

routes = {
    name: Route(name, np.array(counts, dtype=float), fares[name])
    for name, counts in passenger_data.items()
}

summary_rows = []
for name, route in routes.items():
    stats = route.descriptive_statistics()
    summary_rows.append({
        "Route": name,
        "Fare (UGX)": route.fare_ugx,
        "Mean passengers/day": stats["mean"],
        "Population variance": stats["variance"],
        "Population SD": stats["standard_deviation"],
        "10-day gross revenue (UGX)": route.total_revenue(),
        "Mean daily gross revenue (UGX)": float(np.mean(route.daily_revenue())),
    })

route_summary = pd.DataFrame(summary_rows)
route_summary

`Route` stores passenger counts as a NumPy array and provides daily revenue, total revenue and descriptive statistics. Variance and standard deviation use the population definitions from Python's `statistics` module, since the supplied ten days are treated as the period being described.

In [ ]:
fig, ax = plt.subplots()
for name, route in routes.items():
    ax.plot(np.arange(1, 11), route.daily_revenue(), marker="o", label=name)
ax.set(title="Daily gross revenue by route", xlabel="Day", ylabel="Revenue (UGX)")
ax.legend()
plt.show()

## 2. Supply–demand equilibrium for Kampala–Ntinda

In [ ]:
equilibrium_fare, equilibrium_quantity = solve_ntinda_equilibrium()
current_fare = fares["Kampala-Ntinda"]
demand_at_current = 120 - 0.02 * current_fare
supply_at_current = 10 + 0.03 * current_fare

equilibrium_table = pd.DataFrame([
    {"Measure": "Equilibrium fare", "Value": equilibrium_fare, "Unit": "UGX"},
    {"Measure": "Equilibrium quantity", "Value": equilibrium_quantity, "Unit": "passengers/trip-hour"},
    {"Measure": "Current fare", "Value": current_fare, "Unit": "UGX"},
    {"Measure": "Demand at current fare", "Value": demand_at_current, "Unit": "passengers/trip-hour"},
    {"Measure": "Supply at current fare", "Value": supply_at_current, "Unit": "passengers/trip-hour"},
    {"Measure": "Excess demand at current fare", "Value": demand_at_current - supply_at_current, "Unit": "passengers/trip-hour"},
])
equilibrium_table

The equations are rearranged into the linear system

\[
0.02P + Q = 120,\qquad -0.03P + Q = 10.
\]

The solution is a modelled equilibrium fare of **UGX 2,200** and quantity of **76 passengers per trip-hour**. The current fare of UGX 2,000 is UGX 200 below equilibrium. At that fare, the model predicts demand of 80 and supply of 70, an excess demand of 10 passengers per trip-hour. This is a theoretical interpretation of the supplied equations, not a real-world fare recommendation: the assumed curves may not represent actual passenger responses, capacity, regulation or operating costs.

## 3. Rolling-origin backtesting and alpha tuning

In [ ]:
# Forecast days 4-10 (zero-based start_index=3). Every prediction sees only prior days.
mae_rows = []
selected_models = {}
alpha_tuning_rows = []
alpha_grid = np.round(np.arange(0.1, 1.01, 0.1), 2)

for route_name, route in routes.items():
    series = route.passenger_counts
    actual_ma, pred_ma = rolling_origin_predictions(
        series, MovingAverageForecaster(window=3), start_index=3
    )
    actual_trend, pred_trend = rolling_origin_predictions(
        series, LinearTrendForecaster(), start_index=3
    )
    best_alpha, alpha_scores = tune_alpha(series, start_index=3, alphas=alpha_grid)
    actual_ses, pred_ses = rolling_origin_predictions(
        series, ExponentialSmoothingForecaster(alpha=best_alpha), start_index=3
    )

    route_scores = {
        "3-day moving average": mean_absolute_error(actual_ma, pred_ma),
        f"exponential smoothing (α={best_alpha:.1f})": mean_absolute_error(actual_ses, pred_ses),
        "linear trend": mean_absolute_error(actual_trend, pred_trend),
    }
    best_label = min(route_scores, key=route_scores.get)
    selected_models[route_name] = (
        best_label,
        {
            "3-day moving average": MovingAverageForecaster(3),
            f"exponential smoothing (α={best_alpha:.1f})": ExponentialSmoothingForecaster(best_alpha),
            "linear trend": LinearTrendForecaster(),
        }[best_label],
        route_scores[best_label],
        best_alpha,
    )

    for model_name, score in route_scores.items():
        mae_rows.append({"Route": route_name, "Model": model_name, "MAE (passengers)": score})
    for alpha, score in alpha_scores.items():
        alpha_tuning_rows.append({"Route": route_name, "Alpha": alpha, "MAE (passengers)": score})

mae_table = pd.DataFrame(mae_rows)
mae_pivot = mae_table.pivot(index="Route", columns="Model", values="MAE (passengers)")
print("Rolling-origin MAE over days 4–10:")
display(mae_pivot.round(3))
print("Alpha grid-search results:")
display(pd.DataFrame(alpha_tuning_rows).pivot(index="Alpha", columns="Route", values="MAE (passengers)").round(3))
print("Selected model by route:")
display(pd.DataFrame([
    {"Route": name, "Best model": model_info[0], "Selected MAE": model_info[2], "Tuned alpha": model_info[3]}
    for name, model_info in selected_models.items()
]).round(3))

**Method:** each backtest prediction for days 4–10 is made using only earlier observations. MAE is the average absolute difference between forecast and actual passenger counts; lower is better. Exponential-smoothing alpha is selected independently for each route from 0.1 through 1.0 in steps of 0.1, using the same rolling-origin period. Since there are only seven backtest forecasts, model rankings should be treated as provisional rather than conclusive.

## 4. Day-11 passenger and revenue forecasts

In [ ]:
day11_rows = []
day11_passenger_forecasts = {}

for route_name, route in routes.items():
    model_name, model, score, best_alpha = selected_models[route_name]
    passenger_forecast = max(0.0, float(model.predict_next(route.passenger_counts)))
    revenue_forecast = passenger_forecast * route.fare_ugx
    day11_passenger_forecasts[route_name] = passenger_forecast
    day11_rows.append({
        "Route": route_name,
        "Selected model": model_name,
        "Day-11 forecast passengers": passenger_forecast,
        "Fare (UGX)": route.fare_ugx,
        "Day-11 forecast gross revenue (UGX)": revenue_forecast,
    })

day11_table = pd.DataFrame(day11_rows)
display(day11_table.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (route_name, route) in zip(axes, routes.items()):
    days = np.arange(1, 11)
    actual_revenue = route.daily_revenue()
    model_name, model, _, _ = selected_models[route_name]
    forecast_passengers = max(0.0, float(model.predict_next(route.passenger_counts)))
    forecast_revenue = forecast_passengers * route.fare_ugx

    ax.plot(days, actual_revenue, marker="o", label="Actual revenue")
    ax.scatter([11], [forecast_revenue], marker="X", s=100, label="Day-11 forecast")
    ax.plot([10, 11], [actual_revenue[-1], forecast_revenue], linestyle="--", alpha=0.7)
    ax.set_title(route_name)
    ax.set_xlabel("Day")
    ax.set_ylabel("Gross revenue (UGX)")
    ax.ticklabel_format(axis="y", style="plain")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Fleet planning with a 15% buffer

In [ ]:
PASSENGERS_PER_VEHICLE_DAY = 8 * 14  # 8 one-way trips × 14 seats = 112 passenger places
BUFFER = 1.15

fleet_rows = []
for route_name, route in routes.items():
    forecast_passengers = day11_passenger_forecasts[route_name]
    buffered_demand = forecast_passengers * BUFFER
    vehicles = math.ceil(buffered_demand / PASSENGERS_PER_VEHICLE_DAY)
    fleet_rows.append({
        "Route": route_name,
        "Forecast passengers/day": forecast_passengers,
        "Demand with 15% buffer": buffered_demand,
        "Capacity per vehicle/day": PASSENGERS_PER_VEHICLE_DAY,
        "Vehicles to deploy": vehicles,
    })

fleet_table = pd.DataFrame(fleet_rows)
display(fleet_table.round(2))

**Fleet interpretation:** each vehicle provides a nominal 112 passenger places per day (8 one-way trips × 14 passengers). The 15% buffer is applied to the forecast passenger total, then divided by 112 and rounded up because a fraction of a vehicle cannot be deployed. This is a simplified capacity calculation: it assumes all trips can be filled and does not account for direction-specific imbalances, departure-time peaks, downtime, vehicle maintenance or passengers making round trips. If the passenger counts represent boardings that include repeat journeys, the planning interpretation remains passenger boardings rather than unique people.

## 6. Extension: 60-day weekly demand pattern

In [ ]:
# Synthetic daily demand: Monday–Thursday ordinary, Friday busy, Sunday quiet.
# Seeded generator makes the exercise reproducible; these are not real route observations.
rng = np.random.default_rng(2026)
days = np.arange(60)
weekday_pattern = np.array([0, 0, 1, 2, 20, 5, -18], dtype=float)  # Monday=0 ... Sunday=6
weekly_effect = weekday_pattern[days % 7]
synthetic_demand = np.maximum(5, 55 + weekly_effect + rng.normal(0, 3, size=60))

seasonal_model = SeasonalNaiveForecaster(season_length=7)
moving_model = MovingAverageForecaster(window=3)
# Evaluate days 15–60 to give both methods a sufficient history; zero-based start=14.
actual_s, pred_s = rolling_origin_predictions(synthetic_demand, seasonal_model, start_index=14)
actual_m, pred_m = rolling_origin_predictions(synthetic_demand, moving_model, start_index=14)
seasonal_mae = mean_absolute_error(actual_s, pred_s)
moving_mae = mean_absolute_error(actual_m, pred_m)

extension_results = pd.DataFrame([
    {"Model": "Seasonal naive (7-day)", "MAE (passengers)": seasonal_mae},
    {"Model": "3-day moving average", "MAE (passengers)": moving_mae},
])
display(extension_results.round(3))
print("Seasonal naive has lower MAE:", seasonal_mae < moving_mae)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(np.arange(1, 61), synthetic_demand, label="Synthetic actual demand", linewidth=1.5)
ax.set(title="Simulated 60-day demand with weekly pattern", xlabel="Day", ylabel="Passengers/day")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
eval_days = np.arange(15, 61)
ax.plot(eval_days, actual_s, label="Actual")
ax.plot(eval_days, pred_s, label=f"Seasonal naive (MAE={seasonal_mae:.2f})", alpha=0.85)
ax.plot(eval_days, pred_m, label=f"3-day moving average (MAE={moving_mae:.2f})", alpha=0.85)
ax.set(title="Walk-forward forecasts on synthetic weekly data (days 15–60)", xlabel="Day", ylabel="Passengers/day")
ax.legend()
plt.show()

The extension deliberately introduces a weekly cycle: Friday is busier and Sunday is quieter. A seasonal-naïve forecast uses demand from the same weekday one week earlier, while the three-day moving average smooths together neighbouring weekdays. With stable weekly structure and modest random noise, seasonal naïve should generally achieve lower MAE. The displayed scores are the empirical check for this particular seeded simulation; the result does not establish that all Kampala routes have the same weekly pattern.

## Findings & Limitations

The supplied ten-day data show that the three routes have different revenue profiles because passenger volumes and fares differ. Entebbe has the highest fare and is therefore likely to generate the most gross revenue, while Ntinda has the lowest fare. Gross revenue is not profit: the dataset does not include fuel, driver pay, vehicle maintenance, association fees, empty return journeys or other costs. The descriptive statistics quantify the observed ten days but cannot establish long-run demand or special-event effects.

The linear supply–demand equations imply an equilibrium fare of UGX 2,200 and quantity of 76 passengers per trip-hour. The existing Ntinda fare is UGX 200 below that modelled value, and the equations predict excess demand at the current fare. This conclusion depends entirely on the assumed curves and should not be used alone to set real fares.

Forecast model rankings are based on only seven rolling-origin errors (days 4–10). Alpha tuning and choosing the best model on this same short period can overfit the observed sequence; a longer history and a separate validation period would improve confidence. Day-11 revenue estimates assume fares remain fixed and passenger forecasts translate directly into paying boardings. Fleet estimates apply a 15% buffer and assume each 14-seater makes eight fully usable one-way trips daily, ignoring peak-hour constraints and operational downtime. Finally, the 60-day weekly series is synthetic, not observed transport data. It demonstrates why a seasonal model can help when weekly patterns persist, but real deployment decisions should use route-specific historical data and monitor forecast errors over time.